In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from database import get_engine

print("Imports successful!")

Imports successful!


In [2]:
engine = get_engine()

# Load factors
print("Loading factors...")
factors_df = pd.read_sql("""
    SELECT date, ticker, close, daily_return,
           mom_12m, mom_6m, mom_1m, mom_5d, reversal,
           vol_21d, vol_63d, vol_ratio,
           volume_ratio, pvt, dist_from_high, rsi
    FROM factors
    ORDER BY ticker, date
""", engine)
factors_df['date'] = pd.to_datetime(factors_df['date'])

# Load VIX
print("Loading VIX...")
vix_df = pd.read_sql("SELECT * FROM vix ORDER BY date", engine)
vix_df['date'] = pd.to_datetime(vix_df['date'])

# Merge regime into factors
factors_df = factors_df.merge(vix_df[['date', 'vix', 'regime']], 
                               on='date', how='left')
factors_df['regime'] = factors_df['regime'].fillna(0).astype(int)

print(f"Factors loaded: {len(factors_df):,} rows")
print(f"Tickers: {factors_df['ticker'].nunique()}")
print(f"\nRegime distribution in factors:")
print(factors_df['regime'].value_counts().sort_index())

Loading factors...
Loading VIX...
Factors loaded: 1,408,642 rows
Tickers: 577

Regime distribution in factors:
regime
0    978358
1    348832
2     81452
Name: count, dtype: int64


In [4]:
REGIME_WEIGHTS = {
    0: {  # Calm market
        'mom_12m':        0.35,
        'mom_6m':         0.20,
        'vol_21d':       -0.15,
        'volume_ratio':   0.10,
        'dist_from_high':-0.10,
        'rsi':           -0.10
    },
    1: {  # Normal market
        'mom_12m':        0.25,
        'mom_6m':         0.15,
        'vol_21d':       -0.20,
        'volume_ratio':   0.10,
        'dist_from_high':-0.15,
        'rsi':           -0.15
    },
    2: {  # Stressed market
        'mom_12m':        0.10,
        'mom_6m':         0.05,
        'vol_21d':       -0.35,
        'volume_ratio':   0.15,
        'dist_from_high':-0.20,
        'rsi':           -0.15
    }
}

def normalize_factor(series):
    mean = series.mean()
    std  = series.std()
    if std == 0:
        return series * 0
    return (series - mean) / std

def rank_stocks_at_date(factors_df, rebalance_date):
    snapshot = factors_df[factors_df['date'] == rebalance_date].copy()
    
    if len(snapshot) < 10:
        return None
    
    regime  = int(snapshot['regime'].mode()[0])
    weights = REGIME_WEIGHTS[regime]
    
    for factor, weight in weights.items():
        z_col = f'z_{factor}'
        snapshot[z_col] = normalize_factor(snapshot[factor]) * weight
    
    z_cols = [f'z_{f}' for f in weights.keys()]
    snapshot['combined_score'] = snapshot[z_cols].sum(axis=1)
    snapshot['rank']           = snapshot['combined_score'].rank(ascending=False)
    snapshot['regime']         = regime
    
    return snapshot[['date', 'ticker', 'combined_score', 'rank', 'regime']].sort_values('rank')

# Test on first date
def get_monthly_rebalance_dates(df):
    df['year_month'] = df['date'].dt.to_period('M')
    monthly_dates = (
        df.groupby('year_month')['date']
        .max()
        .reset_index()
    )
    monthly_dates.columns = ['year_month', 'rebalance_date']
    return monthly_dates

rebalance_dates = get_monthly_rebalance_dates(factors_df)
test            = rank_stocks_at_date(factors_df, rebalance_dates['rebalance_date'].iloc[0])

print(f"Total rebalancing periods: {len(rebalance_dates)}")
print(f"\nTest ranking for {test['date'].iloc[0].date()} (Regime {test['regime'].iloc[0]}):")
test.head(10)

Total rebalancing periods: 120

Test ranking for 2015-01-30 (Regime 1):


,date,ticker,combined_score,rank,regime
1156016,2015-01-30,SWKS,1.432135,1.0,1
431521,2015-01-30,EW,1.418404,2.0,1
1018662,2015-01-30,PTCT,1.162012,3.0,1
121322,2015-01-30,AVGO,0.847731,4.0,1
690539,2015-01-30,KR,0.828297,5.0,1
799057,2015-01-30,MNST,0.825623,6.0,1
1035460,2015-01-30,RCL,0.821356,7.0,1
739373,2015-01-30,LUV,0.780842,8.0,1
1070670,2015-01-30,ROST,0.742078,9.0,1
651726,2015-01-30,JACK,0.721752,10.0,1


In [5]:
def run_backtest(factors_df, rebalance_dates, top_n=20, bottom_n=20):
    results = []
    
    for i in range(len(rebalance_dates) - 1):
        current_date = rebalance_dates['rebalance_date'].iloc[i]
        next_date    = rebalance_dates['rebalance_date'].iloc[i + 1]
        
        rankings = rank_stocks_at_date(factors_df, current_date)
        if rankings is None:
            continue
        
        regime        = rankings['regime'].iloc[0]
        top_stocks    = rankings[rankings['rank'] <= top_n]['ticker'].tolist()
        bottom_stocks = rankings[rankings['rank'] > len(rankings) - bottom_n]['ticker'].tolist()
        
        current_prices = factors_df[factors_df['date'] == current_date].set_index('ticker')['close']
        next_prices    = factors_df[factors_df['date'] == next_date].set_index('ticker')['close']
        
        top_returns    = []
        bottom_returns = []
        
        for ticker in top_stocks:
            if ticker in current_prices.index and ticker in next_prices.index:
                ret = (next_prices[ticker] - current_prices[ticker]) / current_prices[ticker]
                top_returns.append(ret)
        
        for ticker in bottom_stocks:
            if ticker in current_prices.index and ticker in next_prices.index:
                ret = (next_prices[ticker] - current_prices[ticker]) / current_prices[ticker]
                bottom_returns.append(ret)
        
        spy_current = factors_df[(factors_df['date'] == current_date) & (factors_df['ticker'] == 'SPY')]['close'].values
        spy_next    = factors_df[(factors_df['date'] == next_date)    & (factors_df['ticker'] == 'SPY')]['close'].values
        
        if len(spy_current) > 0 and len(spy_next) > 0:
            spy_return = (spy_next[0] - spy_current[0]) / spy_current[0]
        else:
            spy_return = np.nan
        
        results.append({
            'date':          current_date,
            'next_date':     next_date,
            'regime':        regime,
            'top_return':    np.mean(top_returns) if top_returns else np.nan,
            'bottom_return': np.mean(bottom_returns) if bottom_returns else np.nan,
            'spy_return':    spy_return,
            'top_stocks':    top_stocks,
            'bottom_stocks': bottom_stocks
        })
    
    return pd.DataFrame(results)

print("Running backtest on 577 stocks...")
backtest_results = run_backtest(factors_df, rebalance_dates, top_n=20, bottom_n=20)

backtest_results['cumulative_top']    = (1 + backtest_results['top_return']).cumprod()
backtest_results['cumulative_bottom'] = (1 + backtest_results['bottom_return']).cumprod()
backtest_results['cumulative_spy']    = (1 + backtest_results['spy_return']).cumprod()

print(f"Backtest complete — {len(backtest_results)} periods")
print(f"\nLong Portfolio:  {backtest_results['cumulative_top'].iloc[-1]:.2f}x")
print(f"Short Portfolio: {backtest_results['cumulative_bottom'].iloc[-1]:.2f}x")
print(f"SPY Benchmark:   {backtest_results['cumulative_spy'].iloc[-1]:.2f}x")

Running backtest on 577 stocks...
Backtest complete — 119 periods

Long Portfolio:  8.22x
Short Portfolio: 72.25x
SPY Benchmark:   3.51x


In [6]:
def calculate_performance_metrics(returns, name):
    total   = (1 + returns).cumprod().iloc[-1] - 1
    ann     = (1 + total) ** (1/10) - 1
    vol     = returns.std() * np.sqrt(12)
    sharpe  = ann / vol
    cum     = (1 + returns).cumprod()
    dd      = ((cum - cum.cummax()) / cum.cummax()).min()
    wr      = (returns > 0).mean()

    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(f"  Total Return:      {total*100:.1f}%")
    print(f"  Annualized Return: {ann*100:.1f}%")
    print(f"  Annualized Vol:    {vol*100:.1f}%")
    print(f"  Sharpe Ratio:      {sharpe:.3f}")
    print(f"  Max Drawdown:      {dd*100:.1f}%")
    print(f"  Win Rate:          {wr*100:.1f}%")

calculate_performance_metrics(backtest_results['top_return'],    "LONG PORTFOLIO (Top 20)")
calculate_performance_metrics(backtest_results['bottom_return'], "SHORT PORTFOLIO (Bottom 20)")
calculate_performance_metrics(backtest_results['spy_return'],    "SPY BENCHMARK")


  LONG PORTFOLIO (Top 20)
  Total Return:      722.1%
  Annualized Return: 23.5%
  Annualized Vol:    19.9%
  Sharpe Ratio:      1.180
  Max Drawdown:      -21.8%
  Win Rate:          66.4%

  SHORT PORTFOLIO (Bottom 20)
  Total Return:      7125.2%
  Annualized Return: 53.4%
  Annualized Vol:    501.7%
  Sharpe Ratio:      0.106
  Max Drawdown:      -49.3%
  Win Rate:          56.3%

  SPY BENCHMARK
  Total Return:      251.0%
  Annualized Return: 13.4%
  Annualized Vol:    15.3%
  Sharpe Ratio:      0.873
  Max Drawdown:      -23.9%
  Win Rate:          69.7%
